# Training on a public dataset: bev-decision-150K

> **⚠️ Plan for a long run.** The full fine-tuning section trains every layer of the backbone on
> the full training data, and nothing can be cached when the backbone learns, so it takes far
> longer than head-only training: up to `BACKBONE_EPOCHS` passes through the backbone, forward and
> backward. ModernBERT-base is about a quarter of Qwen3-0.6B's size, so it runs several times
> faster than the decoder version of this notebook, but use a CUDA GPU (on Colab, an L4 or A100
> runtime). Save trained models to Google Drive with `model.save(...)`, since Colab deletes
> `/content` when the runtime disconnects. A run stopped with Ctrl-C keeps its best weights so far
> and continues from where it stopped when `fit` is called again with the same data.

[avbiswas/bev-decision-150K](https://huggingface.co/datasets/avbiswas/bev-decision-150K) holds about
150,000 texts from 18 domains (spatial reasoning, sentiment, retail, support routing, finance, ...),
each with one or more typed questions and their correct answers. It is much harder than the
synthetic tickets in `quickstart.ipynb`: the questions differ from row to row, and many need real
reading comprehension.

This notebook trains ModernBERT-base, a bidirectional encoder, with `tuning="head"` (frozen
backbone, only the decision head learns), then fine-tunes the whole backbone with `tuning="full"`,
and compares both with random guessing on held-out test rows, per question type and per domain. It
runs faster than [`decoder_bev_decision.ipynb`](decoder_bev_decision.ipynb), the same notebook with
Qwen3-0.6B, and the two can be compared directly.

In [ ]:
# On Google Colab, or anywhere sysone isn't installed yet, install it from PyPI.
import importlib.util

if importlib.util.find_spec("sysone") is None:
    %pip install -q "sysone>=0.5"

In [ ]:
import json

from datasets import load_dataset

from sysone import EncoderModel, baselines, compare

N_TRAIN = 3000   # training rows (about 2 questions each); the full train split has 125,614
N_TEST = 1000    # held-out rows from the test split
BACKBONE_EPOCHS = 8  # upper bound for full fine-tuning; early stopping may end it sooner

## Load the data

The correct answer is stored as a `label` inside each question, which sysone reads directly. Each
row's domain is passed as its `category`, so validation is split evenly across domains and
results can be broken down by domain.

In [ ]:
def load_rows(split, n):
    ds = load_dataset("avbiswas/bev-decision-150K", split=split).shuffle(seed=0).select(range(n))
    return [{"state": r["state"], "questions": json.loads(r["questions_json"]), "category": r["domain"]}
            for r in ds]


train = load_rows("train", N_TRAIN)
test = load_rows("test", N_TEST)
print("%d training rows, %d test rows" % (len(train), len(test)))
next(iter(train[0]["questions"].values()))

## Before training: random guessing

Questions differ from row to row here, so the only meaningful baseline is random guessing: a
question with k options is right 1/k of the time.

In [ ]:
results = baselines(test)
compare(results)

## Train

The backbone reads every training question once and its outputs are cached; each epoch then
trains only the head. 10% of rows are held back for validation and early stopping.

In [ ]:
model = EncoderModel(tuning="head")
model.fit(train)

## After training

In [ ]:
results["head"] = model.evaluate(test)
compare(results)

## Fine-tuning the whole backbone

With `tuning="full"`, every layer of ModernBERT trains along with the head. ModernBERT-base is small
enough for this to be affordable, and it is what works for this model: ModernBERT was pretrained only
to fill in masked words, and turning its `[CLS]` and `[MASK]` vectors into a decision means
reshaping the whole network. Small LoRA adapters barely moved it in our runs. (The decoder version
of this notebook uses LoRA and QLoRA, which suit Qwen.)

The backbone's learning rate is chosen for ModernBERT (`backbone_lr="auto"`, 5e-5). Validation runs
about ten times per epoch; training stops once validation loss hasn't improved for 3 epochs, and
the best weights are kept, so `BACKBONE_EPOCHS` is an upper bound.

In [ ]:
full = EncoderModel(tuning="full", epochs=BACKBONE_EPOCHS, batch_size=32)
full.fit(train)
results["full"] = full.evaluate(test)
compare(results)

**Metrics** (from `model.evaluate`, the same for hard and soft labels):
- **accuracy**: the model's most likely option is the right one (for soft labels: the raters'
  top answer). For a score question this means the exact level.
- **distance**: how far the predicted probabilities are from the label (0 = identical, 1 = completely
  different).
- **log_loss**: cross-entropy against the label; lower is better.
- **score_error**: for score questions, the gap between predicted and true expected level.
- **auc**: for yes/no questions, how well the model ranks yes above no (0.5 = chance).
- **brier**: squared distance between the predicted probabilities and the label; lower is better.
- **ece**: expected calibration error, how far "says X% sure" is from "right X% of the time";
  lower is better.

Probabilities are calibrated after training: a temperature per question type, fitted on the
validation slice, softens over-confident answers. This changes **log_loss**, **brier** and **ece**,
never **accuracy**, since the top answer stays the same.

In [ ]:
compare(results, by="overall")

### Calibration

The same model before and after temperature scaling. Accuracy is identical; the probabilities, and
so **log_loss**, **brier** and **ece**, change. `full.report_["calibration"]` has the fitted
temperatures and the validation slice's ECE before and after.

In [ ]:
calibration = {"full": results["full"], "full, before calibration": full.evaluate(test, calibrated=False)}
compare(calibration, by="overall")[["accuracy", "log_loss", "brier", "ece"]]

### Accuracy by domain

The number of test questions per domain is in `results["head"]["by_category"][domain]["n"]`;
small domains give noisy numbers.

In [ ]:
compare(results, by="category").drop(columns="overall").T.sort_values("head", ascending=False)

## What to expect

Training only the head on a frozen backbone is the cheapest setting and the least accurate: the
backbone was never adapted to these questions. Expect clear gains over random guessing, and full
fine-tuning to do much better than head-only on the same rows. Score questions are the hardest. For
them the loss adds an ordinal term (`ordinal_weight`), so predicting a level next to the right one
costs less than one far away; **score_error** shows how far off the predicted level is on average.

Things to try: more training rows (`N_TRAIN`), more `BACKBONE_EPOCHS`, or a larger encoder
(`EncoderModel("answerdotai/ModernBERT-large", tuning="full")`, which needs more GPU memory). A
long run stopped with Ctrl-C continues from where it stopped with `full.fit(train)`.